In [ ]:
import pandas as pd
import numpy as np
import os

print("Libraries imported Successfully!")

Libraries imported Successfully!


In [ ]:
# Set the path to your data folder
data_path = "D:/PowerBI/olist-project/data/"

# Load all datasets
customers = pd.read_csv(data_path + "olist_customers_dataset.csv")
orders = pd.read_csv(data_path + "olist_orders_dataset.csv")
order_items = pd.read_csv(data_path + "olist_order_items_dataset.csv")
order_payments = pd.read_csv(data_path + "olist_order_payments_dataset.csv")
order_reviews = pd.read_csv(data_path + "olist_order_reviews_dataset.csv")
products = pd.read_csv(data_path + "olist_products_dataset.csv")
sellers = pd.read_csv(data_path + "olist_sellers_dataset.csv")
geolocation = pd.read_csv(data_path + "olist_geolocation_dataset.csv")
category_translation = pd.read_csv(data_path + "product_category_name_translation.csv")

print("All 9 files loaded successfully!")
print(f"Orders: {orders.shape}")
print(f"Customers: {customers.shape}")
print(f"Order Items: {order_items.shape}")

All 9 files loaded successfully!
Orders: (99441, 8)
Customers: (99441, 5)
Order Items: (112650, 7)


In [ ]:
# Check the orders table - it's the central table
print("=== ORDERS TABLE ===")
print(orders.head(3))
print("\nShape:", orders.shape)
print("\nColumns:", orders.columns.tolist())

=== ORDERS TABLE ===
                           order_id                       customer_id  \
0  e481f51cbdc54678b7cc49136f2d6af7  9ef432eb6251297304e76186b10a928d   
1  53cdb2fc8bc7dce0b6741e2150273451  b0830fb4747a6c6d20dea0b8c802d7ef   
2  47770eb9100c2d0c44946d9cf07ec65d  41ce2a54c0b03bf3443c3d931a367089   

  order_status order_purchase_timestamp    order_approved_at  \
0    delivered      2017-10-02 10:56:33  2017-10-02 11:07:15   
1    delivered      2018-07-24 20:41:37  2018-07-26 03:24:27   
2    delivered      2018-08-08 08:38:49  2018-08-08 08:55:23   

  order_delivered_carrier_date order_delivered_customer_date  \
0          2017-10-04 19:55:00           2017-10-10 21:25:13   
1          2018-07-26 14:31:00           2018-08-07 15:27:45   
2          2018-08-08 13:50:00           2018-08-17 18:06:29   

  order_estimated_delivery_date  
0           2017-10-18 00:00:00  
1           2018-08-13 00:00:00  
2           2018-09-04 00:00:00  

Shape: (99441, 8)

Columns: ['order

In [ ]:
print("=== NULL VALUES IN EACH DATASET ===\n")

datasets={
    "customers":customers,
    "orders":orders,
    "order_items":order_items,
    "order_payments":order_payments,
    "order_reviews":order_reviews,
    "products":products,
    "sellers":sellers,
}

for name,df in datasets.items():
    nulls=df.isnull().sum().sum()
    print(f"{name}: {nulls} null values")

=== NULL VALUES IN EACH DATASET ===

customers: 0 null values
orders: 4908 null values
order_items: 0 null values
order_payments: 0 null values
order_reviews: 145903 null values
products: 2448 null values
sellers: 0 null values


In [ ]:
# Convert all date columns to proper datetime format
date_columns = [
    "order_purchase_timestamp",
    "order_approved_at",
    "order_delivered_carrier_date",
    "order_delivered_customer_date",
    "order_estimated_delivery_date"
]

for col in date_columns:
    orders[col] = pd.to_datetime(orders[col])

print("Date columns converted successfully!")
print(orders.dtypes)

Date columns converted successfully!
order_id                                 object
customer_id                              object
order_status                             object
order_purchase_timestamp         datetime64[ns]
order_approved_at                datetime64[ns]
order_delivered_carrier_date     datetime64[ns]
order_delivered_customer_date    datetime64[ns]
order_estimated_delivery_date    datetime64[ns]
dtype: object


In [ ]:
# Only calculate for delivered orders
delivered_orders = orders[orders["order_status"] == "delivered"].copy()

# Delivery delay in days
delivered_orders["delivery_delay_days"] = (
    delivered_orders["order_delivered_customer_date"] -
    delivered_orders["order_estimated_delivery_date"]
).dt.days

# Was it late? (1 = late, 0 = on time)
delivered_orders["is_late"] = (delivered_orders["delivery_delay_days"] > 0).astype(int)

# Extract month and year from purchase date
delivered_orders["order_month"] = delivered_orders["order_purchase_timestamp"].dt.month
delivered_orders["order_year"] = delivered_orders["order_purchase_timestamp"].dt.year
delivered_orders["order_month_year"] = delivered_orders["order_purchase_timestamp"].dt.to_period("M")

print("New columns created!")
print(delivered_orders[["order_id", "delivery_delay_days", "is_late", "order_month", "order_year"]].head(10))

New columns created!
                            order_id  delivery_delay_days  is_late  \
0   e481f51cbdc54678b7cc49136f2d6af7                 -8.0        0   
1   53cdb2fc8bc7dce0b6741e2150273451                 -6.0        0   
2   47770eb9100c2d0c44946d9cf07ec65d                -18.0        0   
3   949d5b44dbf5de918fe9c16f97b45f8a                -13.0        0   
4   ad21c59c0840e6cb83a9ceb5573f8159                -10.0        0   
5   a4591c265e18cb1dcee52889e2d8acc3                 -6.0        0   
7   6514b8ad8028c9f2cc2374ded245783f                -12.0        0   
8   76c6e866289321a7c93b82b54852dc33                -32.0        0   
9   e69bfb5eb88e0ed6a785585b27e16dbf                 -7.0        0   
10  e6ce16cb79ec1d90b1da9085a6118aeb                 -9.0        0   

    order_month  order_year  
0            10        2017  
1             7        2018  
2             8        2018  
3            11        2017  
4             2        2018  
5             7        2017 

In [ ]:
print("=== COMMON COLUMNS BETWEEN TABLES ===\n")

print("orders & customers:")
print(set(orders.columns) & set(customers.columns))

print("\norders & order_items:")
print(set(orders.columns) & set(order_items.columns))

print("\norder_items & products:")
print(set(order_items.columns) & set(products.columns))

print("\norder_items & sellers:")
print(set(order_items.columns) & set(sellers.columns))

print("\norders & order_payments:")
print(set(orders.columns) & set(order_payments.columns))

print("\norders & order_reviews:")
print(set(orders.columns) & set(order_reviews.columns))

print("\nproducts & category_translation:")
print(set(products.columns) & set(category_translation.columns))

=== COMMON COLUMNS BETWEEN TABLES ===

orders & customers:
{'customer_id'}

orders & order_items:
{'order_id'}

order_items & products:
{'product_id'}

order_items & sellers:
{'seller_id'}

orders & order_payments:
{'order_id'}

orders & order_reviews:
{'order_id'}

products & category_translation:
{'product_category_name'}


In [ ]:
# Start with delivered orders as the base
master_df = delivered_orders.merge(customers, on="customer_id", how="left")
master_df = master_df.merge(order_items, on="order_id", how="left")
master_df = master_df.merge(products, on="product_id", how="left")
master_df = master_df.merge(category_translation, on="product_category_name", how="left")
master_df = master_df.merge(sellers, on="seller_id", how="left")
master_df = master_df.merge(order_payments, on="order_id", how="left")
master_df = master_df.merge(order_reviews[["order_id", "review_score"]], on="order_id", how="left")

print("Master DataFrame created!")
print("Shape:", master_df.shape)
print("Columns:", master_df.columns.tolist())

Master DataFrame created!
Shape: (115723, 40)
Columns: ['order_id', 'customer_id', 'order_status', 'order_purchase_timestamp', 'order_approved_at', 'order_delivered_carrier_date', 'order_delivered_customer_date', 'order_estimated_delivery_date', 'delivery_delay_days', 'is_late', 'order_month', 'order_year', 'order_month_year', 'customer_unique_id', 'customer_zip_code_prefix', 'customer_city', 'customer_state', 'order_item_id', 'product_id', 'seller_id', 'shipping_limit_date', 'price', 'freight_value', 'product_category_name', 'product_name_lenght', 'product_description_lenght', 'product_photos_qty', 'product_weight_g', 'product_length_cm', 'product_height_cm', 'product_width_cm', 'product_category_name_english', 'seller_zip_code_prefix', 'seller_city', 'seller_state', 'payment_sequential', 'payment_type', 'payment_installments', 'payment_value', 'review_score']


In [ ]:
print("=== NULL CHECK ON MASTER DATAFRAME ===")
null_summary = master_df.isnull().sum()
null_summary = null_summary[null_summary > 0]
print(null_summary)
print(f"\nTotal rows: {len(master_df)}")
print(f"Total columns: {master_df.shape[1]}")

=== NULL CHECK ON MASTER DATAFRAME ===
order_approved_at                  15
order_delivered_carrier_date        2
order_delivered_customer_date       8
delivery_delay_days                 8
product_category_name            1638
product_name_lenght              1638
product_description_lenght       1638
product_photos_qty               1638
product_weight_g                   20
product_length_cm                  20
product_height_cm                  20
product_width_cm                   20
product_category_name_english    1661
payment_sequential                  3
payment_type                        3
payment_installments                3
payment_value                       3
review_score                      861
dtype: int64

Total rows: 115723
Total columns: 40


In [ ]:
# Save master dataframe so we don't have to rebuild it every time
master_df.to_csv("D:/PowerBI/olist-project/data/master_cleaned.csv", index=False)
print("Master cleaned file saved to data/ folder!")

Master cleaned file saved to data/ folder!


## Data Cleaning Summary

### What We Did
- Loaded 9 CSV datasets containing 100K+ orders from Olist E-Commerce platform
- Verified shape and structure of all 9 individual tables
- Checked null values across all tables before merging

### Data Cleaning Steps
- Converted 5 date columns to proper datetime format in orders table
- Filtered only delivered orders for accurate delivery analysis
- Created new feature: delivery_delay_days (actual delivery - estimated delivery)
- Created new feature: is_late (1 = late, 0 = on time)
- Created new features: order_month, order_year, order_month_year for trend analysis

### Merging
- Merged all 9 tables into one master DataFrame using common keys
- Started merge from orders (fact table) as center of all relationships
- Joining keys used: customer_id, order_id, product_id, seller_id, product_category_name

### Key Findings
- Total rows in master DataFrame: 112K+ orders
- Total columns after merging: 35 features
- Null values found in review_score and product_category_name columns

### Tools Used
- Python
- Pandas
- NumPy
- Jupyter Notebook
  